# UMAP Embedding Of Graph Vectorizer Features

Thin notebook for selecting Task A or Task B, vectorizing persisted protein contact networks, projecting the feature matrix to 2D with UMAP, and plotting the result colored by class label.


## 1. Configure

In [1]:
try:
    from _bootstrap import configure_paths
except ImportError:
    from notebooks._bootstrap import configure_paths

PROJECT_ROOT = configure_paths(
    include=("abstractgraph_graphicalizer", "simplex_pair_vectorizer", "abstractgraph", "abstractgraph_ml")
)

from enzyme_classification.modeling.datasets import load_graph_dataset_from_status
from enzyme_classification.modeling.vectorizers import make_graph_vectorizer
from abstractgraph.utils import plot_embedding_2d

try:
    from nsppk import NSPPK
except ImportError:
    NSPPK = None

try:
    from umap import UMAP
except ImportError as error:
    raise ImportError("Install notebook dependencies with: pip install -e '.[notebooks]'") from error

import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import TruncatedSVD
import pandas as pd


In [2]:
TASK = "B"  # "A" for enzyme vs non-enzyme, "B" for first-level EC class
TRANSFORMER_KIND = "simplex_pair"  # "simplex_pair" or "nsppk"
NODE_LABEL_ALPHABET = "aa20"  # "aa20", "hp2", "charge4", "physchem5", "chem7", or "dayhoff6"
EDGE_DISTANCE_THRESHOLDS = None  # None => "contact"; e.g. [5.0] => "contact_close"/"contact_far".
RANDOM_STATE = 42
SAMPLE_LIMIT = None  # Set to an integer for quick smoke tests, or None for all graphs.

NSPPK_KWARGS = {
    "radius": 1,
    "distance": 4,
    "connector": 1,
    "nbits": 14,
    "dense": False,
    "parallel": True,
}
SIMPLEX_PAIR_KWARGS = {
    "max_simplex_dimension": 2,
    "min_distance": 0,
    "max_distance": 4,
    "distance_mode": "representative",
    "distance_jitter": "auto",
    "pairing_mode": "rooted",
    "nbits": 14,
    "dense": False,
    "include_self": True,
    "cache_graphs": False,
    "n_jobs": -1,
}

PCA_N_COMPONENTS = 100

# UMAP spread controls:
# - lower n_neighbors emphasizes smaller local neighborhoods;
# - higher min_dist prevents points from packing tightly inside clusters;
# - higher spread gives the embedding more room overall;
# - higher repulsion_strength pushes unrelated points farther apart;
# - higher negative_sample_rate makes that repulsion estimate stronger but slower.
UMAP_N_NEIGHBORS = 5
UMAP_MIN_DIST = 0.95
UMAP_METRIC = "cosine"
UMAP_SPREAD = 5.0
UMAP_REPULSION_STRENGTH = 5.0
UMAP_NEGATIVE_SAMPLE_RATE = 20
UMAP_INIT = "spectral"

# Plot-only spread for dense piles. This does not change the UMAP embedding values.
PLOT_COORD_SCALE = 1.6
PLOT_JITTER = 0.04
PLOT_POINT_SIZE = 18
PLOT_POINT_ALPHA = 0.45

NEIGHBORHOOD_UNION_K_VALUES = [5, 11]
NEIGHBORHOOD_UNION_ALPHA = 0.55
NEIGHBORHOOD_UNION_Z = 1
NEIGHBORHOOD_UNION_QUANTILE = 1.0

TASK_A_STATUS = PROJECT_ROOT / "data/processed/task_a/graph_build_status.tsv"
TASK_B_STATUS = PROJECT_ROOT / "data/processed/task_b/graph_build_status.tsv"
STATUS_PATH = TASK_A_STATUS if TASK.upper() == "A" else TASK_B_STATUS
STATUS_PATH


PosixPath('/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/EnzymeClassification/data/processed/task_b/graph_build_status.tsv')

## 2. Load Graphs

In [3]:
dataset = load_graph_dataset_from_status(STATUS_PATH, task=TASK)
graphs = dataset.graphs
y = dataset.labels
metadata = dataset.metadata

if SAMPLE_LIMIT is not None:
    graphs = graphs[:SAMPLE_LIMIT]
    y = y[:SAMPLE_LIMIT]
    metadata = metadata.iloc[:SAMPLE_LIMIT].reset_index(drop=True)

print(f"task={TASK}")
print(f"status_path={STATUS_PATH}")
print(f"graphs={len(graphs)} labels={len(y)}")
pd.Series(y, name="target").value_counts().sort_index()


task=B
status_path=/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/EnzymeClassification/data/processed/task_b/graph_build_status.tsv
graphs=15332 labels=15332


target
1    2462
2    5149
3    6388
4     679
5     509
6     145
Name: count, dtype: int64

## 3. Vectorize Graphs

In [4]:
vectorizer = make_graph_vectorizer(
    transformer_kind=TRANSFORMER_KIND,
    node_label_alphabet=NODE_LABEL_ALPHABET,
    edge_distance_thresholds=EDGE_DISTANCE_THRESHOLDS,
    nsppk_class=NSPPK,
    simplex_pair_kwargs=SIMPLEX_PAIR_KWARGS,
    nsppk_kwargs=NSPPK_KWARGS,
)

features = vectorizer.fit_transform(graphs)
print(f"features_shape={features.shape}")
print(f"features_nnz={getattr(features, 'nnz', 'dense')}")
vectorizer


features_shape=(15332, 16384)
features_nnz=220216840


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('protein_label_graphicalizer', ...), ('graph_vectorizer', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,alphabet,'aa20'
,node_label_key,'label'
,edge_label_key,'label'
,edge_distance_key,'distance'
,edge_distance_thresholds,None
,unknown_policy,'keep'
,unknown_label,'unknown'


## 4. 2D Projection

In [ ]:
if len(graphs) < 2:
    raise ValueError("Need at least two graphs for a 2D embedding.")

pca_n_components = min(PCA_N_COMPONENTS, features.shape[0] - 1, features.shape[1] - 1)
if pca_n_components < 2:
    raise ValueError("Need at least two PCA components before UMAP.")

pca_model = TruncatedSVD(
    n_components=pca_n_components,
    random_state=RANDOM_STATE,
)
pca_features = pca_model.fit_transform(features)
print(f"pca_features_shape={pca_features.shape}")
print(f"pca_explained_variance_ratio_sum={pca_model.explained_variance_ratio_.sum():.4f}")

umap_model = UMAP(
    n_components=2,
    n_neighbors=min(UMAP_N_NEIGHBORS, max(2, len(graphs) - 1)),
    min_dist=UMAP_MIN_DIST,
    metric=UMAP_METRIC,
    spread=UMAP_SPREAD,
    repulsion_strength=UMAP_REPULSION_STRENGTH,
    negative_sample_rate=UMAP_NEGATIVE_SAMPLE_RATE,
    init=UMAP_INIT,
    random_state=RANDOM_STATE,
)
embedding = umap_model.fit_transform(pca_features)

embedding_df = metadata[["sample_id", "pdb_id", "protein_name", "first_level_ec_numbers", "target"]].copy()
embedding_df["label"] = y
embedding_df["umap_1"] = embedding[:, 0]
embedding_df["umap_2"] = embedding[:, 1]
embedding_df.head()


pca_features_shape=(15332, 100)
pca_explained_variance_ratio_sum=0.9662


/home/fabrizio/.venvs/py313/lib/python3.13/site-packages/umap/umap_.py:1401: RuntimeWarning: divide by zero encountered in power
  return 1.0 / (1.0 + a * x ** (2 * b))
/home/fabrizio/.venvs/py313/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/fabrizio/.venvs/py313/lib/python3.13/site-packages/umap/spectral.py:548: UserWarning: Spectral initialisation failed! The eigenvector solver
failed. This is likely due to too small an eigengap. Consider
adding some noise or jitter to your data.

Falling back to random initialisation!
  warn(
/home/fabrizio/.venvs/py313/lib/python3.13/site-packages/umap/spectral.py:548: UserWarning: Spectral initialisation failed! The eigenvector solver
failed. This is likely due to too small an eigengap. Consider
adding some noise or jitter to your data.

Falling back to random initialisation!
  warn(


## 5. Plot

In [ ]:
label_order = sorted(pd.Series(y).unique())
color_map = plt.get_cmap("tab10" if len(label_order) <= 10 else "tab20")
label_to_color = {label: color_map(idx % color_map.N) for idx, label in enumerate(label_order)}
plot_df = embedding_df.copy()
plot_df["umap_1"] = plot_df["umap_1"] * PLOT_COORD_SCALE
plot_df["umap_2"] = plot_df["umap_2"] * PLOT_COORD_SCALE
if PLOT_JITTER:
    jitter_rng = np.random.default_rng(RANDOM_STATE)
    jitter = jitter_rng.uniform(-PLOT_JITTER, PLOT_JITTER, size=(len(plot_df), 2))
    plot_df["umap_1_plot"] = plot_df["umap_1"] + jitter[:, 0]
    plot_df["umap_2_plot"] = plot_df["umap_2"] + jitter[:, 1]
else:
    plot_df["umap_1_plot"] = plot_df["umap_1"]
    plot_df["umap_2_plot"] = plot_df["umap_2"]

fig, ax = plt.subplots(figsize=(8, 6), dpi=130)
for label in label_order:
    rows = plot_df[plot_df["label"] == label]
    ax.scatter(
        rows["umap_1_plot"],
        rows["umap_2_plot"],
        label=str(label),
        s=PLOT_POINT_SIZE,
        alpha=PLOT_POINT_ALPHA,
        color=label_to_color[label],
        linewidths=0.25,
        edgecolors="white",
    )

ax.set_title(
    f"Task {TASK} {TRANSFORMER_KIND}, alphabet={NODE_LABEL_ALPHABET}, "
    f"min_dist={UMAP_MIN_DIST}, spread={UMAP_SPREAD}, repulsion={UMAP_REPULSION_STRENGTH}, "
    f"scale={PLOT_COORD_SCALE}"
)
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.legend(title="label", bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()


## 6. Neighborhood Union Overlay

Use the AbstractGraph `plot_embedding_2d` helper to visualize k-nearest-neighbor class-union regions on the same 2D embedding.


In [ ]:
label_order = sorted(pd.Series(y).unique())
label_categories = pd.Categorical(y, categories=label_order)
label_codes = label_categories.codes
label_code_map = pd.DataFrame(
    {
        "code": range(len(label_categories.categories)),
        "label": [str(label) for label in label_categories.categories],
    }
)
display(label_code_map)
size = 10
Z_for_union = embedding_df[["umap_1", "umap_2"]].to_numpy() * PLOT_COORD_SCALE
fig, axes = plt.subplots(
    1,
    1 + len(NEIGHBORHOOD_UNION_K_VALUES),
    figsize=(size * (1 + len(NEIGHBORHOOD_UNION_K_VALUES)), size),
    constrained_layout=True,
)
axes = np.atleast_1d(axes)

plot_embedding_2d(
    vectorizer,
    graphs,
    label_codes,
    title_prefix="Scatter",
    mode="scatter",
    alpha=PLOT_POINT_ALPHA,
    ax=axes[0],
    show=False,
    Z=Z_for_union,
    quantile=NEIGHBORHOOD_UNION_QUANTILE,
)
for ax, k in zip(axes[1:], NEIGHBORHOOD_UNION_K_VALUES):
    plot_embedding_2d(
        vectorizer,
        graphs,
        label_codes,
        title_prefix=f"kNN class union k={k}",
        mode="knn_class_union",
        k=k,
        alpha=NEIGHBORHOOD_UNION_ALPHA,
        z=NEIGHBORHOOD_UNION_Z,
        ax=ax,
        show=False,
        show_instances=True,
        Z=Z_for_union,
        quantile=NEIGHBORHOOD_UNION_QUANTILE,
    )
plt.show()


## 7. Inspect Points

In [ ]:
embedding_df.sort_values(["label", "sample_id"]).head(30)